**Voice-to-Grasp - fine-tuning of DistilBERT and RoBERTa** on the grasp-command dataset
(dataset.json + updated_dataset.json, 7 classes; paper Section 2.3). Produces the model folder
`bert_finetuned_updated/` used in Experiments 2-4. Outputs of this notebook were cleared for the release.

In [ ]:
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
print(f"Torch version : {torch.__version__}")
print(f"GPU Available : {torch.cuda.is_available()}")
print(f"CPU Threads : {torch.get_num_threads()}")

In [ ]:
dataset_json = pd.read_json("dataset.jsonl", lines=True)
# print(dataset.head())

train_text, test_text, train_labels, test_labels = train_test_split(
    dataset_json["text"],
    dataset_json["label"],
    test_size=0.2,
    random_state=42,
    stratify=dataset_json["label"]
)

train_df = pd.DataFrame({"text" : train_text, "label" : train_labels})
test_df = pd.DataFrame({"text" : test_text, "label" : test_labels})

# print(test_df.head())

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train" : train_dataset,
    "test" : test_dataset
})

# print(dataset)

In [ ]:
# model_checkpoint = "roberta-base"
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
# def preprocessed_text(examples):
#     return tokenizer(examples["text"], truncation=True, padding=True,  return_tensors="pt")
# tokenized_dataset = dataset.map(preprocessed_text, batched=True)


def preprocessed_text(examples):
    tokenized = tokenizer(examples["text"], truncation=True, padding=True)
    tokenized["labels"] = examples["label"]
    return tokenized
tokenized_dataset = dataset.map(preprocessed_text, batched=True)


In [ ]:
num_labels= dataset_json["label"].value_counts().__len__()
# print(num_labels)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels
)

training_args = TrainingArguments(
    # output_dir = "./results",
    output_dir = "./bert_finetuned_updated",  
    eval_strategy="epoch" , 
    save_strategy = "epoch",
    learning_rate= 2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True
)

In [ ]:
def compute_metrics(pred):
    labels=pred.label_ids
    preds=np.argmax(pred.predictions, axis=1)
    acc=accuracy_score(labels,preds)
    return {"accuracy" : acc}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args= training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

acc = accuracy_score(labels, preds)
print(f"Test accuracy = {acc:4f}")

In [ ]:
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(3,2))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
plt.xlabel("predictes")
plt.ylabel("True")
plt.title("confusion Matrix")
plt.show()

In [ ]:
print(classification_report(labels, preds))

In [ ]:
# model.save_pretrained("./grasp_intent_model")
# tokenizer.save_pretrained("./grasp_intent_model")
model.save_pretrained("./bert_finetuned_updated")
tokenizer.save_pretrained("./bert_finetuned_updated")